# Телематика бортовых систем в системе менеджмента качества автомобилей

Выявление актуальных факторов дефектов (жалобы на шум) и разграничение статистической и технологической значимости.

Данные — `data/data.csv`, эксплуатационные данные автомобилей сервисной сети. Признаки — модель автомобиля, пробег, температура наружного воздуха, флаг рекламации на шум.

Ноутбук идёт по шагам исследования. Шаг 1 — однофакторные и двухфакторные доли рекламаций. Шаг 2 — сравнение распределений пробега и температуры (Манн–Уитни, величина эффекта). Шаг 3 — устойчивые оценки, ловушка большой выборки, пороги инженерной значимости и экономика. Шаг 4 — рисунки статьи.

## Шаг 1. Однофакторный и двухфакторный анализ долей рекламаций

Данные загружаются, пробег и температура разбиваются на интервалы, считаются доли рекламаций по модели, пробегу и температуре и по их парам. Проверяется сопряженность (χ², V Крамера).

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from scipy import stats


def main():
    # 1. Загрузка данных
    csv_path = 'data/data.csv'
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Файл {csv_path} не найден!")
    
    df = pd.read_csv(csv_path)
    print("=" * 70)
    print("1. АУДИТ И СТРУКТУРА ДАННЫХ")
    print("=" * 70)
    print(f"Размерность датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
    print("\nПропущенные значения:")
    print(df.isnull().sum())
    
    n_total = len(df)
    n_noise = int(df['noise'].sum())
    base_rate = (n_noise / n_total) * 100
    print(f"\nОбщий объем выборки: {n_total}")
    print(f"Число рекламаций на шум: {n_noise} ({base_rate:.2f}%)")
    print(f"Число авто без шума: {n_total - n_noise} ({100 - base_rate:.2f}%)")
    
    # 2. Логический биннинг непрерывных переменных
    # Пробег (предметно-эксплуатационные интервалы)
    mileage_bins = [0, 80000, 95000, 105000, 120000, 200000]
    mileage_labels = ['до 80 000 км (минимальный)', 
                      '80 000 – 95 000 км (умеренный)', 
                      '95 000 – 105 000 км (средний)', 
                      '105 000 – 120 000 км (повышенный)', 
                      'свыше 120 000 км (высокий)']
    df['mileage_bin'] = pd.cut(df['mileage'], bins=mileage_bins, labels=mileage_labels)
    
    # Температура (климатические зоны)
    temp_bins = [-50, 0, 15, 25, 60]
    temp_labels = ['< 0 °C (Мороз)', 
                   '0 … +15 °C (Прохладно)', 
                   '+15 … +25 °C (Комфорт)', 
                   '> +25 °C (Жара)']
    df['temp_bin'] = pd.cut(df['temperature'], bins=temp_bins, labels=temp_labels)
    
    # 3. Однофакторный анализ
    print("\n" + "=" * 70)
    print("2. ОДНОФАКТОРНЫЙ АНАЛИЗ ДОЛИ РЕКЛАМАЦИЙ")
    print("=" * 70)
    
    factors = [
        ('model', 'Модель автомобиля'),
        ('mileage_bin', 'Диапазон пробега'),
        ('temp_bin', 'Климатическая зона')
    ]
    
    for col, name in factors:
        agg = df.groupby(col, observed=False).agg(
            total=('noise', 'count'),
            noise_cnt=('noise', 'sum')
        )
        agg['no_noise_cnt'] = agg['total'] - agg['noise_cnt']
        agg['noise_pct'] = (agg['noise_cnt'] / agg['total']) * 100
        agg['share_of_fleet'] = (agg['total'] / n_total) * 100
        agg['relative_risk'] = agg['noise_pct'] / base_rate
        
        ct = pd.crosstab(df[col], df['noise'])
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        cramers_v = np.sqrt(chi2 / (len(df) * (min(ct.shape) - 1)))
        
        print(f"\n--- Фактор: {name} ({col}) ---")
        display_df = agg[['total', 'share_of_fleet', 'noise_cnt', 'no_noise_cnt', 'noise_pct', 'relative_risk']].copy()
        display_df.columns = ['Всего', 'Доля авто (%)', 'С шумом', 'Без шума', 'Доля шума (%)', 'RR к базе']
        print(display_df.round(2).to_string())
        print(f"Критерий Пирсона: chi2 = {chi2:.2f}, df = {dof}, p-value = {p:.4e}, V Крамера = {cramers_v:.4f}")

    # 4. Двухфакторный анализ (пересечение факторов)
    print("\n" + "=" * 70)
    print("3. ДВУХФАКТОРНЫЙ АНАЛИЗ (ПЕРЕСЕЧЕНИЕ ФАКТОРОВ)")
    print("=" * 70)
    
    cross_pairs = [
        ('model', 'temp_bin', '«Модель + Температура»'),
        ('model', 'mileage_bin', '«Модель + Пробег»'),
        ('temp_bin', 'mileage_bin', '«Температура + Пробег»')
    ]
    
    comb_records = []
    
    for r_col, c_col, title in cross_pairs:
        pt_cnt = df.pivot_table(index=r_col, columns=c_col, values='noise', aggfunc='count', observed=False)
        pt_noise = df.pivot_table(index=r_col, columns=c_col, values='noise', aggfunc='sum', observed=False)
        pt_rate = (pt_noise / pt_cnt * 100).round(2)
        
        group_series = df[r_col].astype(str) + " x " + df[c_col].astype(str)
        ct = pd.crosstab(group_series, df['noise'])
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        cramers_v = np.sqrt(chi2 / (len(df) * (min(ct.shape) - 1)))
        
        print(f"\n--- Пересечение {title} ---")
        print("Доля рекламаций на шум (%):")
        print(pt_rate.to_string())
        print(f"Сопряженность: chi2 = {chi2:.2f}, df = {dof}, p-value = {p:.4e}, V Крамера = {cramers_v:.4f}")
        
        for r in pt_cnt.index:
            for c in pt_cnt.columns:
                tot = int(pt_cnt.loc[r, c])
                nc = int(pt_noise.loc[r, c])
                pct = round((nc / tot) * 100, 2) if tot > 0 else 0.0
                rr = round(pct / base_rate, 2)
                comb_records.append({
                    'Pair': title,
                    'Combination': f"{r} + {c}",
                    'Total': tot,
                    'Noise': nc,
                    'Noise_Pct': pct,
                    'RR': rr
                })
                
    # 5. Поиск аномальных комбинаций
    print("\n" + "=" * 70)
    print("4. РАНЖИРОВАНИЕ И АНОМАЛЬНЫЕ КОМБИНАЦИИ")
    print("=" * 70)
    
    combs_df = pd.DataFrame(comb_records)
    top10 = combs_df.sort_values(by='Noise_Pct', ascending=False).head(10)
    print("ТОП-10 комбинаций с максимальной долей рекламаций:")
    print(top10.to_string(index=False))

main()

1. АУДИТ И СТРУКТУРА ДАННЫХ
Размерность датасета: 11345 строк, 4 столбцов

Пропущенные значения:
model          0
mileage        0
temperature    0
noise          0
dtype: int64

Общий объем выборки: 11345
Число рекламаций на шум: 1833 (16.16%)
Число авто без шума: 9512 (83.84%)

2. ОДНОФАКТОРНЫЙ АНАЛИЗ ДОЛИ РЕКЛАМАЦИЙ

--- Фактор: Модель автомобиля (model) ---
       Всего  Доля авто (%)  С шумом  Без шума  Доля шума (%)  RR к базе
model                                                                   
M01     5733          50.53      695      5038          12.12       0.75
M11     3374          29.74      867      2507          25.70       1.59
M21     2238          19.73      271      1967          12.11       0.75
Критерий Пирсона: chi2 = 322.61, df = 2, p-value = 8.8506e-71, V Крамера = 0.1686

--- Фактор: Диапазон пробега (mileage_bin) ---
                                   Всего  Доля авто (%)  С шумом  Без шума  Доля шума (%)  RR к базе
mileage_bin                             

## Шаг 2. Сравнение распределений пробега и температуры

Для автомобилей с рекламацией и без неё сравниваются распределения пробега и температуры. Критерий Манна–Уитни, d Коэна, ранговая бисериальная корреляция и вероятность превосходства (CLES).

In [2]:
import sys
import os
import pandas as pd
import numpy as np
from scipy import stats


def analyze_variable(df, var_name, var_label, unit):
    print("=" * 80)
    print(f"СТАТИСТИЧЕСКИЙ АНАЛИЗ ПЕРЕМЕННОЙ: {var_label.upper()} ({var_name}), [{unit}]")
    print("=" * 80)
    
    g0 = df[df['noise'] == 0][var_name].values
    g1 = df[df['noise'] == 1][var_name].values
    all_vals = df[var_name].values
    
    n0 = len(g0)
    n1 = len(g1)
    N = len(all_vals)
    
    # 1. Описательные статистики
    m0, m1, m_all = np.mean(g0), np.mean(g1), np.mean(all_vals)
    std0, std1, std_all = np.std(g0, ddof=1), np.std(g1, ddof=1), np.std(all_vals, ddof=1)
    se0, se1 = std0 / np.sqrt(n0), std1 / np.sqrt(n1)
    
    med0, med1, med_all = np.median(g0), np.median(g1), np.median(all_vals)
    q25_0, q75_0 = np.percentile(g0, 25), np.percentile(g0, 75)
    q25_1, q75_1 = np.percentile(g1, 25), np.percentile(g1, 75)
    iqr0, iqr1 = q75_0 - q25_0, q75_1 - q25_1
    
    min0, max0 = np.min(g0), np.max(g0)
    min1, max1 = np.min(g1), np.max(g1)
    
    skew0, skew1 = stats.skew(g0), stats.skew(g1)
    kurt0, kurt1 = stats.kurtosis(g0), stats.kurtosis(g1)
    
    # Тест на нормальность Д'Агостино-Пирсона (подходит для больших N)
    k2_0, p_norm0 = stats.normaltest(g0)
    k2_1, p_norm1 = stats.normaltest(g1)
    
    print(f"1. ОПИСАТЕЛЬНЫЕ СТАТИСТИКИ:")
    print(f"   Группа 'Нет шума' (n0={n0}):")
    print(f"      Mean = {m0:.2f} {unit}, SE = {se0:.2f}, Std = {std0:.2f}")
    print(f"      Median = {med0:.2f} {unit}, IQR = {iqr0:.2f} [Q1={q25_0:.2f}, Q3={q75_0:.2f}]")
    print(f"      Min = {min0:.2f}, Max = {max0:.2f}, Размах = {max0 - min0:.2f}")
    print(f"      Асимметрия (Skewness) = {skew0:.4f}, Эксцесс (Kurtosis) = {kurt0:.4f}")
    print(f"      Тест Д'Агостино на нормальность: K2={k2_0:.2f}, p={p_norm0:.4e}")
    
    print(f"\n   Группа 'Есть шум' (n1={n1}):")
    print(f"      Mean = {m1:.2f} {unit}, SE = {se1:.2f}, Std = {std1:.2f}")
    print(f"      Median = {med1:.2f} {unit}, IQR = {iqr1:.2f} [Q1={q25_1:.2f}, Q3={q75_1:.2f}]")
    print(f"      Min = {min1:.2f}, Max = {max1:.2f}, Размах = {max1 - min1:.2f}")
    print(f"      Асимметрия (Skewness) = {skew1:.4f}, Эксцесс (Kurtosis) = {kurt1:.4f}")
    print(f"      Тест Д'Агостино на нормальность: K2={k2_1:.2f}, p={p_norm1:.4e}")
    
    diff_mean = m1 - m0
    diff_med = med1 - med0
    print(f"\n   Разности центральных тенденций:")
    print(f"      Разность средних (m1 - m0) = {diff_mean:+.2f} {unit} ({diff_mean/m0*100:+.3f}%)")
    print(f"      Разность медиан (med1 - med0) = {diff_med:+.2f} {unit} ({diff_med/med0*100:+.3f}%)")
    
    # 2. Критерий Манна-Уитни (scipy.stats.mannwhitneyu)
    # mannwhitneyu(g1, g0, alternative='two-sided')
    res_mwu = stats.mannwhitneyu(g1, g0, alternative='two-sided')
    u1 = res_mwu.statistic
    u0 = n0 * n1 - u1
    p_val = res_mwu.pvalue
    
    # Односторонние альтернативы
    res_greater = stats.mannwhitneyu(g1, g0, alternative='greater')
    res_less = stats.mannwhitneyu(g1, g0, alternative='less')
    
    # Математическое ожидание и дисперсия U при H0
    mu_u = n0 * n1 / 2.0
    _, tie_counts = np.unique(np.concatenate([g1, g0]), return_counts=True)
    tie_term = np.sum(tie_counts**3 - tie_counts) / ((n0 + n1) * (n0 + n1 - 1))
    var_u = (n0 * n1 / 12.0) * ((n0 + n1 + 1) - tie_term)
    std_u = np.sqrt(var_u)
    z_stat = (u1 - mu_u) / std_u
    
    # Величины эффекта (Effect Size)
    r_rb = (u1 - u0) / (n0 * n1)
    r_effect = z_stat / np.sqrt(N)
    cles = u1 / (n0 * n1)
    
    # Cohen's d (параметрический размер эффекта)
    s_pooled = np.sqrt(((n0 - 1) * std0**2 + (n1 - 1) * std1**2) / (n0 + n1 - 2))
    cohen_d = (m1 - m0) / s_pooled
    
    # Hedges' g
    j = 1 - (3 / (4 * (n0 + n1) - 9))
    hedges_g = cohen_d * j
    
    # t-тест Уэлча для сравнения
    ttest_res = stats.ttest_ind(g1, g0, equal_var=False)
    
    print("\n2. РЕЗУЛЬТАТЫ КРИТЕРИЯ МАННА-УИТНИ (Mann-Whitney U):")
    print(f"   U1 (шум) = {u1:.1f}")
    print(f"   U0 (нет шума) = {u0:.1f}")
    print(f"   E(U) = {mu_u:.1f}, Std(U) = {std_u:.2f}")
    print(f"   Z-score = {z_stat:.4f}")
    print(f"   p-value (двусторонний, two-sided) = {p_val:.6f} ({p_val:.4e})")
    print(f"   p-value (H1: шум > нет шума, greater) = {res_greater.pvalue:.6f} ({res_greater.pvalue:.4e})")
    print(f"   p-value (H1: шум < нет шума, less)    = {res_less.pvalue:.6f} ({res_less.pvalue:.4e})")
    
    print("\n3. ВЕЛИЧИНЫ ЭФФЕКТА (EFFECT SIZES):")
    print(f"   Rank-Biserial correlation (r_rb) = {r_rb:.5f}")
    print(f"   Нормированный размер эффекта r (|Z| / sqrt(N)) = {abs(r_effect):.5f}")
    print(f"   Вероятность превосходства (CLES, P(X_шум > X_без_шума)) = {cles:.4f} ({cles*100:.2f}%)")
    print(f"   Cohen's d = {cohen_d:.5f}")
    print(f"   Hedges' g = {hedges_g:.5f}")
    print(f"   Справочно: t-тест Уэлча: t = {ttest_res.statistic:.4f}, p = {ttest_res.pvalue:.6f}")
    
    if abs(cohen_d) < 0.2:
        effect_interp = "Пренебрежимо малый (negligible, < 0.2)"
    elif abs(cohen_d) < 0.5:
        effect_interp = "Малый (small, 0.2 - 0.5)"
    elif abs(cohen_d) < 0.8:
        effect_interp = "Средний (medium, 0.5 - 0.8)"
    else:
        effect_interp = "Крупный (large, >= 0.8)"
    print(f"   Интерпретация эффекта по шкале Коэна: {effect_interp}")

    # 4. Стратифицированный анализ по моделям (M01, M11, M21)
    print("\n4. СТРАТИФИЦИРОВАННЫЙ АНАЛИЗ ПО МОДЕЛЯМ АВТОМОБИЛЕЙ:")
    for model in sorted(df['model'].unique()):
        sub_df = df[df['model'] == model]
        sub_g0 = sub_df[sub_df['noise'] == 0][var_name].values
        sub_g1 = sub_df[sub_df['noise'] == 1][var_name].values
        
        sub_mwu = stats.mannwhitneyu(sub_g1, sub_g0, alternative='two-sided')
        sub_d = (np.mean(sub_g1) - np.mean(sub_g0)) / np.sqrt(((len(sub_g0)-1)*np.var(sub_g0, ddof=1) + (len(sub_g1)-1)*np.var(sub_g1, ddof=1))/(len(sub_g0)+len(sub_g1)-2))
        
        print(f"   Модель {model}: n0={len(sub_g0)}, n1={len(sub_g1)} | Mean_0={np.mean(sub_g0):.2f}, Mean_1={np.mean(sub_g1):.2f}, Delta={np.mean(sub_g1)-np.mean(sub_g0):+.2f} | MWU U={sub_mwu.statistic:.1f}, p={sub_mwu.pvalue:.4f}, Cohen d={sub_d:+.4f}")
    
    return {
        'var_name': var_name,
        'var_label': var_label,
        'unit': unit,
        'n0': n0, 'n1': n1, 'N': N,
        'm0': m0, 'm1': m1, 'diff_mean': diff_mean,
        'med0': med0, 'med1': med1, 'diff_med': diff_med,
        'std0': std0, 'std1': std1,
        'iqr0': iqr0, 'iqr1': iqr1,
        'u1': u1, 'u0': u0, 'z_stat': z_stat,
        'p_val': p_val,
        'r_rb': r_rb, 'r_effect': r_effect, 'cles': cles,
        'cohen_d': cohen_d, 'hedges_g': hedges_g
    }

def main():
    csv_path = 'data/data.csv'
    df = pd.read_csv(csv_path)
    
    print("ДАТАСЕТ ЗАГРУЖЕН УСПЕШНО. ОБЩИЙ РАЗМЕР: ", df.shape)
    
    res_mileage = analyze_variable(df, 'mileage', 'Пробег (наработка)', 'км')
    res_temp = analyze_variable(df, 'temperature', 'Температура наружного воздуха', '°C')
    
    print("\n" + "=" * 80)
    print("ИТОГОВОЕ СВОДНОЕ СРАВНЕНИЕ ДВУХ ФАКТОРОВ:")
    print("=" * 80)
    print(f"{'Параметр':<30} | {'Пробег (mileage)':<22} | {'Температура (temperature)':<22}")
    print("-" * 80)
    s_n_mileage = f"{res_mileage['n0']} / {res_mileage['n1']}"
    s_n_temp = f"{res_temp['n0']} / {res_temp['n1']}"
    s_dm_mileage = f"{res_mileage['diff_mean']:+.1f} км"
    s_dm_temp = f"{res_temp['diff_mean']:+.3f} °C"
    s_dmed_mileage = f"{res_mileage['diff_med']:+.1f} км"
    s_dmed_temp = f"{res_temp['diff_med']:+.3f} °C"
    s_u_mileage = f"{res_mileage['u1']:.1f}"
    s_u_temp = f"{res_temp['u1']:.1f}"
    s_z_mileage = f"{res_mileage['z_stat']:+.4f}"
    s_z_temp = f"{res_temp['z_stat']:+.4f}"
    s_p_mileage = f"{res_mileage['p_val']:.5f}"
    s_p_temp = f"{res_temp['p_val']:.5f}"
    s_d_mileage = f"{res_mileage['cohen_d']:+.4f}"
    s_d_temp = f"{res_temp['cohen_d']:+.4f}"
    s_r_mileage = f"{res_mileage['r_rb']:+.4f}"
    s_r_temp = f"{res_temp['r_rb']:+.4f}"
    s_c_mileage = f"{res_mileage['cles']*100:.2f}%"
    s_c_temp = f"{res_temp['cles']*100:.2f}%"

    print(f"{'Объем выборки (N)':<30} | {res_mileage['N']:<22} | {res_temp['N']:<22}")
    print(f"{'Размер групп (n0 / n1)':<30} | {s_n_mileage:<22} | {s_n_temp:<22}")
    print(f"{'Сдвиг средних (Mean1 - Mean0)':<30} | {s_dm_mileage:<22} | {s_dm_temp:<22}")
    print(f"{'Сдвиг медиан (Med1 - Med0)':<30} | {s_dmed_mileage:<22} | {s_dmed_temp:<22}")
    print(f"{'U-статистика (U1)':<30} | {s_u_mileage:<22} | {s_u_temp:<22}")
    print(f"{'Z-статистика':<30} | {s_z_mileage:<22} | {s_z_temp:<22}")
    print(f"{'p-value (Манн-Уитни)':<30} | {s_p_mileage:<22} | {s_p_temp:<22}")
    print(f"{'Величина эффекта Cohen d':<30} | {s_d_mileage:<22} | {s_d_temp:<22}")
    print(f"{'Rank-Biserial r_rb':<30} | {s_r_mileage:<22} | {s_r_temp:<22}")
    print(f"{'CLES (Вероятность превосход.)':<30} | {s_c_mileage:<22} | {s_c_temp:<22}")

main()


ДАТАСЕТ ЗАГРУЖЕН УСПЕШНО. ОБЩИЙ РАЗМЕР:  (11345, 4)
СТАТИСТИЧЕСКИЙ АНАЛИЗ ПЕРЕМЕННОЙ: ПРОБЕГ (НАРАБОТКА) (mileage), [км]
1. ОПИСАТЕЛЬНЫЕ СТАТИСТИКИ:
   Группа 'Нет шума' (n0=9512):
      Mean = 100027.50 км, SE = 103.14, Std = 10059.31
      Median = 100060.50 км, IQR = 13785.50 [Q1=93137.00, Q3=106922.50]
      Min = 59354.00, Max = 137775.00, Размах = 78421.00
      Асимметрия (Skewness) = -0.0064, Эксцесс (Kurtosis) = 0.0228
      Тест Д'Агостино на нормальность: K2=0.30, p=8.5976e-01

   Группа 'Есть шум' (n1=1833):
      Mean = 100493.10 км, SE = 228.90, Std = 9799.89
      Median = 100477.00 км, IQR = 13174.00 [Q1=93936.00, Q3=107110.00]
      Min = 66128.00, Max = 135137.00, Размах = 69009.00
      Асимметрия (Skewness) = -0.0387, Эксцесс (Kurtosis) = 0.0241
      Тест Д'Агостино на нормальность: K2=0.55, p=7.6032e-01

   Разности центральных тенденций:
      Разность средних (m1 - m0) = +465.60 км (+0.465%)
      Разность медиан (med1 - med0) = +416.50 км (+0.416%)

2. РЕЗУЛЬТА

## Шаг 3. Устойчивые оценки, ловушка большой выборки и экономика

Бутстрап-интервалы сдвигов, оценка Ходжеса–Лемана, моделирование зависимости p-value от объёма выборки при фиксированном эффекте, пороги инженерной значимости и экономическое сравнение стратегий.

In [3]:
import sys
import os
import pandas as pd
import numpy as np
from scipy import stats


# Загрузка данных
data_path = "data/data.csv"
df = pd.read_csv(data_path)
print(f"Загружен датасет: N = {len(df)}")

g0_mileage = df[df['noise'] == 0]['mileage'].values
g1_mileage = df[df['noise'] == 1]['mileage'].values

g0_temp = df[df['noise'] == 0]['temperature'].values
g1_temp = df[df['noise'] == 1]['temperature'].values

n0 = len(g0_mileage)
n1 = len(g1_mileage)
N = n0 + n1

# 1. Бутстрап доверительных интервалов разности медиан и средних
np.random.seed(42)
B = 2000

def bootstrap_diffs(g0, g1, B=2000):
    diff_means = np.empty(B)
    diff_medians = np.empty(B)
    n0_len = len(g0)
    n1_len = len(g1)
    for i in range(B):
        s0 = np.random.choice(g0, size=n0_len, replace=True)
        s1 = np.random.choice(g1, size=n1_len, replace=True)
        diff_means[i] = np.mean(s1) - np.mean(s0)
        diff_medians[i] = np.median(s1) - np.median(s0)
    
    ci_mean = np.percentile(diff_means, [2.5, 97.5])
    ci_median = np.percentile(diff_medians, [2.5, 97.5])
    return ci_mean, ci_median

print("Расчет бутстрап доверительных интервалов (B=2000)...")
ci_mean_mil, ci_med_mil = bootstrap_diffs(g0_mileage, g1_mileage, B)
ci_mean_tmp, ci_med_tmp = bootstrap_diffs(g0_temp, g1_temp, B)

# Оценка Ходжеса-Лемана (псевдомедиана попарных разностей) методом субсэмплинга (из-за размера N=11345)
def hodges_lehmann_subsample(g0, g1, sub_size=1800, n_iter=30):
    hl_estimates = []
    for _ in range(n_iter):
        sub0 = np.random.choice(g0, size=sub_size, replace=False)
        sub1 = np.random.choice(g1, size=min(sub_size, len(g1)), replace=False)
        diffs = sub1[:, None] - sub0[None, :]
        hl_estimates.append(np.median(diffs))
    return np.mean(hl_estimates), np.percentile(hl_estimates, [2.5, 97.5])

hl_mil, hl_mil_ci = hodges_lehmann_subsample(g0_mileage, g1_mileage, sub_size=1800, n_iter=30)
hl_tmp, hl_tmp_ci = hodges_lehmann_subsample(g0_temp, g1_temp, sub_size=1800, n_iter=30)

print("=" * 80)
print("1. ОЦЕНКИ СДВИГА И РАЗМЕРА ЭФФЕКТА (EFFECT SIZE):")
print("=" * 80)
print("ПРОБЕГ (mileage):")
print(f"  Медиана без шума: {np.median(g0_mileage):.1f} км, с шумом: {np.median(g1_mileage):.1f} км")
print(f"  Сдвиг медиан: {np.median(g1_mileage) - np.median(g0_mileage):+.1f} км ({(np.median(g1_mileage) - np.median(g0_mileage))/np.median(g0_mileage)*100:+.3f}%)")
print(f"  95% ДИ сдвига медиан (бутстрап): [{ci_med_mil[0]:.1f}, {ci_med_mil[1]:.1f}] км")
print(f"  Сдвиг средних: {np.mean(g1_mileage) - np.mean(g0_mileage):+.1f} км ({(np.mean(g1_mileage) - np.mean(g0_mileage))/np.mean(g0_mileage)*100:+.3f}%)")
print(f"  95% ДИ сдвига средних (бутстрап): [{ci_mean_mil[0]:.1f}, {ci_mean_mil[1]:.1f}] км")
print(f"  Оценка Ходжеса-Лемана (псевдомедиана): {hl_mil:+.1f} км (95% ДИ: [{hl_mil_ci[0]:.1f}, {hl_mil_ci[1]:.1f}])")
print(f"  Cohen's d = 0.0465, CLES = 51.36%, Rank-Biserial r = 0.0272")

print("\nТЕМПЕРАТУРА (temperature):")
print(f"  Медиана без шума: {np.median(g0_temp):.2f} °C, с шумом: {np.median(g1_temp):.2f} °C")
print(f"  Сдвиг медиан: {np.median(g1_temp) - np.median(g0_temp):+.2f} °C ({(np.median(g1_temp) - np.median(g0_temp))/np.median(g0_temp)*100:+.3f}%)")
print(f"  95% ДИ сдвига медиан (бутстрап): [{ci_med_tmp[0]:.2f}, {ci_med_tmp[1]:.2f}] °C")
print(f"  Сдвиг средних: {np.mean(g1_temp) - np.mean(g0_temp):+.3f} °C ({(np.mean(g1_temp) - np.mean(g0_temp))/np.mean(g0_temp)*100:+.3f}%)")
print(f"  95% ДИ сдвига средних (бутстрап): [{ci_mean_tmp[0]:.3f}, {ci_mean_tmp[1]:.3f}] °C")
print(f"  Оценка Ходжеса-Лемана (псевдомедиана): {hl_tmp:+.3f} °C (95% ДИ: [{hl_tmp_ci[0]:.3f}, {hl_tmp_ci[1]:.3f}])")
print(f"  Cohen's d = 0.0026, CLES = 50.26%, Rank-Biserial r = 0.0052")

# 2. Моделирование зависимости p-value от размера выборки N (Large Sample Size Fallacy)
print("\n" + "=" * 80)
print("2. ДЕМОНСТРАЦИЯ ЛОВУШКИ СВЕРХБОЛЬШОЙ ВЫБОРКИ (LARGE SAMPLE SIZE FALLACY):")
print("=" * 80)
print("При сохранении неизменных физических параметров распределений:")
print(f"   Пробег: delta = +465.6 км, s_pooled = 10018 км (Cohen d = 0.0465)")
print(f"   Температура: delta = +0.026 °C, s_pooled = 9.94 °C (Cohen d = 0.0026)")
print("\nТаблица зависимости p-value от размера совокупности N (сохраняя пропорцию n1/N = 16.16%):")
print("--------------------------------------------------------------------------------")
print("N выборки  | p-val (пробег)  | Вывод по пробегу      | p-val (темп.)  | Вывод по темп.")
print("--------------------------------------------------------------------------------")
test_Ns = [1000, 5000, 11345, 20000, 50000, 100000, 500000]
d_mil = 0.04648
d_tmp = 0.00264
prop1 = n1 / N
prop0 = n0 / N

for tN in test_Ns:
    tn0 = tN * prop0
    tn1 = tN * prop1
    se_diff = np.sqrt(1/tn0 + 1/tn1)
    z_mil = d_mil / se_diff
    p_mil = 2 * (1 - stats.norm.cdf(abs(z_mil)))
    z_tmp = d_tmp / se_diff
    p_tmp = 2 * (1 - stats.norm.cdf(abs(z_tmp)))
    
    dec_mil = "Значимо (p<0.05)" if p_mil < 0.05 else "НЕ значимо"
    dec_tmp = "Значимо (p<0.05)" if p_tmp < 0.05 else "НЕ значимо"
    print(f"{tN:<10} | {p_mil:<15.6f} | {dec_mil:<21} | {p_tmp:<14.6f} | {dec_tmp}")

print("--------------------------------------------------------------------------------")
print("ВЫВОД: При N > 20 000 даже крошечный сдвиг пробега в 465 км становится формально")
print("'высокодостоверным' (p < 0.01), хотя инженерно это всего 465 км из 100 000 км!")

# 3. Расчет порогов инженерной значимости
print("\n" + "=" * 80)
print("3. ПОРОГИ ИНЖЕНЕРНОЙ / ПРАКТИЧЕСКОЙ ЗНАЧИМОСТИ:")
print("=" * 80)
print("Фактор Пробега:")
print("   - Регламентный межсервисный интервал ТО: 15 000 км")
print("   - Погрешность одометра и износа протектора шин (1.5-2%): ~1 500 - 2 000 км")
print("   - Фактический сдвиг медиан: 416.5 км")
print("   - Отношение факт. сдвига к интервалу ТО: 416.5 / 15 000 = 2.78% (неразличимо)")
print("   - Отношение факт. сдвига к погрешности одометра: 416.5 / 1 500 = 27.8% (ниже шума измерений)")
print("\nФактор Температуры:")
print("   - Инструментальная погрешность бортового термометра: +/- 0.5 ... 1.0 °C")
print("   - Сезонный суточный перепад температур: 10 ... 15 °C")
print("   - Фактический сдвиг медиан: -0.10 °C (средних: +0.026 °C)")
print("   - Фактический сдвиг меньше погрешности датчика в: 0.5 / 0.026 = 19.2 раза!")

# 4. Анализ модели и формулирование точных экономических оценок
print("\n" + "=" * 80)
print("4. ЭКОНОМИЧЕСКАЯ МОДЕЛЬ: СТОИМОСТЬ ОШИБОЧНЫХ VS ЦЕЛЕВЫХ РЕШЕНИЙ:")
print("=" * 80)

cost_part_replace = 12000 # руб.
cost_r_and_d = 5000000 # руб.

cars_near_100k = len(df[(df['mileage'] >= 80000) & (df['mileage'] <= 120000)]) # ~10838 авто
cost_scenario_1 = cars_near_100k * cost_part_replace
print(f"Сценарий 1 (Ложная сервисная кампания на ТО при пробеге ~100 тыс км):")
print(f"   Охват: {cars_near_100k} авто")
print(f"   Затраты: {cost_scenario_1:,.0f} руб. (~{cost_scenario_1/1e6:.1f} млн руб.)")
print(f"   Результат: Напрасная замена на M01 и M21, дефект на M11 НЕ устранен на конвейере.")

cost_scenario_2 = 15000000 # руб.
print(f"\nСценарий 2 (Ложная смена эластомера на заводе для всех моделей):")
print(f"   Затраты: ~15-20 млн руб.")
print(f"   Результат: Первопричина дефекта M11 (аэродинамический профиль/зазоры) не устранена.")

cost_scenario_3 = cost_r_and_d + len(df[(df['model']=='M11') & (df['noise']==1)]) * cost_part_replace
print(f"\nСценарий 3 (Адресное научно обоснованное решение):")
print(f"   Затраты завода на устранение конструктивного дефекта M11: {cost_r_and_d:,.0f} руб.")
print(f"   Затраты сервиса на устранение обращений M11: {len(df[(df['model']=='M11') & (df['noise']==1)]) * cost_part_replace:,.0f} руб.")
print(f"   Итоговые затраты: {cost_scenario_3:,.0f} руб. (~{cost_scenario_3/1e6:.1f} млн руб.)")
print(f"   Экономия по сравнению с ложной кампанией ТО: {cost_scenario_1 - cost_scenario_3:,.0f} руб. (~{(cost_scenario_1 - cost_scenario_3)/1e6:.1f} млн руб.)")
print(f"   Главный эффект: 100% ликвидация системной причины брака на конвейере!")


Загружен датасет: N = 11345


Расчет бутстрап доверительных интервалов (B=2000)...


1. ОЦЕНКИ СДВИГА И РАЗМЕРА ЭФФЕКТА (EFFECT SIZE):
ПРОБЕГ (mileage):
  Медиана без шума: 100060.5 км, с шумом: 100477.0 км
  Сдвиг медиан: +416.5 км (+0.416%)
  95% ДИ сдвига медиан (бутстрап): [-186.0, 979.6] км
  Сдвиг средних: +465.6 км (+0.465%)
  95% ДИ сдвига средних (бутстрап): [-40.0, 947.3] км
  Оценка Ходжеса-Лемана (псевдомедиана): +464.1 км (95% ДИ: [171.0, 870.9])
  Cohen's d = 0.0465, CLES = 51.36%, Rank-Biserial r = 0.0272

ТЕМПЕРАТУРА (temperature):
  Медиана без шума: 15.10 °C, с шумом: 15.00 °C
  Сдвиг медиан: -0.10 °C (-0.662%)
  95% ДИ сдвига медиан (бутстрап): [-0.70, 0.50] °C
  Сдвиг средних: +0.026 °C (+0.175%)
  95% ДИ сдвига средних (бутстрап): [-0.475, 0.539] °C
  Оценка Ходжеса-Лемана (псевдомедиана): +0.097 °C (95% ДИ: [-0.200, 0.400])
  Cohen's d = 0.0026, CLES = 50.26%, Rank-Biserial r = 0.0052

2. ДЕМОНСТРАЦИЯ ЛОВУШКИ СВЕРХБОЛЬШОЙ ВЫБОРКИ (LARGE SAMPLE SIZE FALLACY):
При сохранении неизменных физических параметров распределений:
   Пробег: delta = +465.6 к

## Шаг 4. Рисунки статьи

Общие настройки оформления и подготовка данных для рисунков. Рисунки сохраняются в папку `pictures` в разрешении 300 dpi.

In [4]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats


# -----------------------------------------------------------------------------
# НАСТРОЙКА АКАДЕМИЧЕСКОГО СТИЛЯ ОФОРМЛЕНИЯ
# -----------------------------------------------------------------------------
plt.rcParams.update({
    'font.sans-serif': ['Segoe UI', 'DejaVu Sans', 'Arial', 'Helvetica'],
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 11.5,
    'axes.titlesize': 12.5,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 13.5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'grid.color': '#E0E0E0',
    'grid.linestyle': '--',
    'grid.linewidth': 0.6,
    'grid.alpha': 0.8,
})

COLOR_NO_NOISE = '#3274A1'   # Сдержанный синий / Slate Blue
COLOR_NOISE = '#C0392B'      # Академический кирпично-красный
COLOR_BASELINE = '#2C3E50'   # Темно-графитовый
COLOR_BOX_LINE = '#2C3E50'   # Темный контур

OUTPUT_DIR = 'pictures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# -----------------------------------------------------------------------------
data_path = "data/data.csv"
if not os.path.exists(data_path):
    data_path = "car_service (1).csv"

df = pd.read_csv(data_path)
N_TOTAL = len(df)
N_NOISE = int(df['noise'].sum())
P_BASE = (N_NOISE / N_TOTAL) * 100.0

print(f"Успешно загружен массив данных: N = {N_TOTAL}, число рекламаций = {N_NOISE} ({P_BASE:.2f}%)")

# Логический биннинг
mileage_bins = [0, 80000, 95000, 105000, 120000, 200000]
mileage_labels = ['до 80 тыс. км', '80–95 тыс. км', '95–105 тыс. км', '105–120 тыс. км', 'свыше 120 тыс. км']
df['mileage_bin'] = pd.cut(df['mileage'], bins=mileage_bins, labels=mileage_labels)

temp_bins = [-50, 0, 15, 25, 60]
temp_labels = ['< 0 °C (Мороз)', '0 … +15 °C (Прохладно)', '+15 … +25 °C (Комфорт)', '> +25 °C (Жара)']
df['temp_bin'] = pd.cut(df['temperature'], bins=temp_bins, labels=temp_labels)

# Функция для расчета интервала Вильсона для биномиальной доли
def wilson_score_interval(count, nobs, alpha=0.05):
    z = stats.norm.ppf(1 - alpha / 2)
    p = count / nobs
    denom = 1 + z**2 / nobs
    center = (p + z**2 / (2 * nobs)) / denom
    spread = z * np.sqrt(p * (1 - p) / nobs + z**2 / (4 * nobs**2)) / denom
    lower = max(0.0, center - spread) * 100
    upper = min(1.0, center + spread) * 100
    return lower, upper


os.makedirs(OUTPUT_DIR, exist_ok=True)

Успешно загружен массив данных: N = 11345, число рекламаций = 1833 (16.16%)


### Рис. 1. Частота рекламаций на шум по моделям автопарка

In [5]:
# -----------------------------------------------------------------------------
# ГРАФИК 1: СТОЛБЧАТАЯ ДИАГРАММА ЧАСТОТЫ ДЕФЕКТОВ ПО МОДЕЛЯМ АВТОМОБИЛЕЙ
# -----------------------------------------------------------------------------
print("Генерация Графика 1: Столбчатая диаграмма частоты дефектов по моделям...")
fig, ax = plt.subplots(figsize=(8.8, 5.8))

model_stats = []
models = sorted(df['model'].unique())
for m in models:
    sub = df[df['model'] == m]
    tot = len(sub)
    ns = sub['noise'].sum()
    rate = (ns / tot) * 100
    ci_low, ci_high = wilson_score_interval(ns, tot)
    rr = rate / P_BASE
    model_stats.append({
        'model': m,
        'total': tot,
        'noise': ns,
        'rate': rate,
        'err_low': rate - ci_low,
        'err_high': ci_high - rate,
        'rr': rr
    })

m_df = pd.DataFrame(model_stats)

x_pos = np.arange(len(m_df))
bars = ax.bar(
    x_pos, m_df['rate'],
    yerr=[m_df['err_low'], m_df['err_high']],
    capsize=6,
    width=0.52,
    color=['#4A7BB0' if r < 20 else '#D9534F' for r in m_df['rate']],
    edgecolor='#2C3E50',
    linewidth=1.1,
    alpha=0.92,
    zorder=3
)

# Базовая линия среднего по автопарку
ax.axhline(P_BASE, color='#2C3E50', linestyle='--', linewidth=1.5, zorder=2,
           label=f'Базовый уровень автопарка: {P_BASE:.2f}% (N = {N_TOTAL:,})'.replace(',', ' '))

# Подписи значений над столбцами с плашками для защиты от наложения линий
for bar, row in zip(bars, m_df.itertuples()):
    y_val = bar.get_height()
    err = row.err_high
    label_text = f"{row.rate:.2f}%\n({row.noise:,} / {row.total:,} авт.)\nRR = {row.rr:.2f}".replace(',', ' ')
    
    # Смещение по y с учетом плашки
    y_target = y_val + err + 1.2
    
    ax.annotate(
        label_text,
        xy=(bar.get_x() + bar.get_width() / 2, y_target),
        ha='center', va='bottom',
        fontsize=9.2, fontweight='bold' if row.rate > 20 else 'normal',
        color='#900C3F' if row.rate > 20 else '#2C3E50',
        bbox=dict(boxstyle='round,pad=0.25', facecolor='#FFFFFF', edgecolor='#BDC3C7', alpha=0.95, lw=0.7),
        zorder=5
    )

# Статистический блок теста Пирсона
chi2_val, p_val, dof, _ = stats.chi2_contingency(pd.crosstab(df['model'], df['noise']))
cramer_v = np.sqrt(chi2_val / (N_TOTAL * (2 - 1)))

stats_text = (
    f"Критерий $\chi^2$ Пирсона = {chi2_val:.2f} (df = {dof})\n"
    f"$p$-value < 10$^{{-70}}$ (экстремально значимо)\n"
    f"$V$ Крамера = {cramer_v:.4f} (умеренно-сильная связь)"
)
ax.text(
    0.03, 0.95, stats_text,
    transform=ax.transAxes,
    fontsize=9.2, verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#BDC3C7', alpha=0.95)
)

# Выделение критической аномалии модели M11
ax.annotate(
    "Аномальный рост дефектности:\nв 2.12 раза выше M01 и M21!\n(Относительный риск RR = 1.59)",
    xy=(1, m_df.loc[m_df['model'] == 'M11', 'rate'].values[0] / 2),
    xytext=(1.45, 23.5),
    arrowprops=dict(facecolor='#C0392B', shrink=0.08, width=1.5, headwidth=7),
    fontsize=9.2, fontweight='bold', color='#C0392B',
    bbox=dict(boxstyle='square,pad=0.4', facecolor='#FDEDEC', edgecolor='#E6B0AA', alpha=0.95),
    zorder=6
)

ax.set_ylabel('Частота рекламаций на акустический шум, %')
ax.set_xlabel('Модель автомобиля')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"Модель {m}" for m in m_df['model']], fontweight='bold')
ax.set_ylim(0, 36)
ax.grid(axis='y', zorder=0)
ax.legend(loc='upper right', frameon=True, facecolor='#FFFFFF', edgecolor='#BDC3C7')

plt.tight_layout()
fig1_path = os.path.join(OUTPUT_DIR, 'fig1_defect_frequency_by_model.png')
plt.savefig(fig1_path)
plt.close()
print(f"Сохранен: {fig1_path}")


Генерация Графика 1: Столбчатая диаграмма частоты дефектов по моделям...


<>:69: SyntaxWarning: invalid escape sequence '\c'
<>:69: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_907/3338308097.py:69: SyntaxWarning: invalid escape sequence '\c'
  f"Критерий $\chi^2$ Пирсона = {chi2_val:.2f} (df = {dof})\n"


Сохранен: pictures/fig1_defect_frequency_by_model.png


### Рис. 2. Сравнение пробега и температуры для групп парка

In [6]:
# -----------------------------------------------------------------------------
# ГРАФИК 2: BOXPLOT СО СДВИГОМ СРЕДНИХ, МЕДИАН И ОТОБРАЖЕНИЕМ P-VALUE
# -----------------------------------------------------------------------------
print("Генерация Графика 2: Boxplot распределений пробега и температуры...")

g0_mil = df[df['noise'] == 0]['mileage'].values / 1000.0  # в тыс. км
g1_mil = df[df['noise'] == 1]['mileage'].values / 1000.0

g0_tmp = df[df['noise'] == 0]['temperature'].values
g1_tmp = df[df['noise'] == 1]['temperature'].values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.0, 6.4))

box_colors = ['#4A90E2', '#E74C3C']

# 1. Пробег
bp1 = ax1.boxplot(
    [g0_mil, g1_mil],
    positions=[1, 2],
    widths=0.48,
    patch_artist=True,
    showmeans=True,
    meanline=True,
    meanprops=dict(color='#F39C12', linewidth=2.0, linestyle='--'),
    medianprops=dict(color='#2C3E50', linewidth=2.2),
    boxprops=dict(linewidth=1.2),
    whiskerprops=dict(color='#2C3E50', linewidth=1.2),
    capprops=dict(color='#2C3E50', linewidth=1.2),
    flierprops=dict(marker='o', markersize=2.5, markerfacecolor='#95A5A6', alpha=0.35, markeredgecolor='none')
)

for patch, color in zip(bp1['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
    patch.set_edgecolor('#2C3E50')

mwu_mil_two = stats.mannwhitneyu(g1_mil * 1000, g0_mil * 1000, alternative='two-sided')
mwu_mil_one = stats.mannwhitneyu(g1_mil * 1000, g0_mil * 1000, alternative='greater')
d_mil = (np.mean(g1_mil) - np.mean(g0_mil)) / np.sqrt((np.var(g0_mil, ddof=1) + np.var(g1_mil, ddof=1)) / 2)
delta_me_mil = (np.median(g1_mil) - np.median(g0_mil)) * 1000
delta_mu_mil = (np.mean(g1_mil) - np.mean(g0_mil)) * 1000

y_max1 = 140
h1 = 2.5
ax1.plot([1, 1, 2, 2], [y_max1, y_max1 + h1, y_max1 + h1, y_max1], lw=1.2, c='#2C3E50')
ax1.text(1.5, y_max1 + h1 + 1.2, f"$p = {mwu_mil_two.pvalue:.3f}$ (различия незначимы)",
         ha='center', va='bottom', color='#2C3E50', fontweight='bold', fontsize=11.0)

ax1.set_xticks([1, 2])
ax1.set_xticklabels(['Без шума\n($noise = 0$)', 'С шумом\n($noise = 1$)'], fontweight='bold', fontsize=11.0)
ax1.set_ylabel('Наработка (пробег), тыс. км', fontsize=12.0)
ax1.set_title('(а) Наработка (пробег), тыс. км', pad=12, fontweight='bold', fontsize=12.5)
ax1.set_ylim(60, 155)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# 2. Температура
bp2 = ax2.boxplot(
    [g0_tmp, g1_tmp],
    positions=[1, 2],
    widths=0.48,
    patch_artist=True,
    showmeans=True,
    meanline=True,
    meanprops=dict(color='#F39C12', linewidth=2.0, linestyle='--'),
    medianprops=dict(color='#2C3E50', linewidth=2.2),
    boxprops=dict(linewidth=1.2),
    whiskerprops=dict(color='#2C3E50', linewidth=1.2),
    capprops=dict(color='#2C3E50', linewidth=1.2),
    flierprops=dict(marker='o', markersize=2.5, markerfacecolor='#95A5A6', alpha=0.35, markeredgecolor='none')
)

for patch, color in zip(bp2['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
    patch.set_edgecolor('#2C3E50')

mwu_tmp = stats.mannwhitneyu(g1_tmp, g0_tmp, alternative='two-sided')

y_max2 = 45
h2 = 2.0
ax2.plot([1, 1, 2, 2], [y_max2, y_max2 + h2, y_max2 + h2, y_max2], lw=1.2, c='#2C3E50')
ax2.text(1.5, y_max2 + h2 + 1.2, f"$p = {mwu_tmp.pvalue:.3f}$ (различия незначимы)",
         ha='center', va='bottom', color='#2C3E50', fontweight='bold', fontsize=11.0)

ax2.set_xticks([1, 2])
ax2.set_xticklabels(['Без шума\n($noise = 0$)', 'С шумом\n($noise = 1$)'], fontweight='bold', fontsize=10.5)
ax2.set_ylabel('Температура наружного воздуха, °C', fontsize=11.5)
ax2.set_title('(б) Температура наружного воздуха, °C', pad=12, fontweight='bold', fontsize=12)
ax2.set_ylim(-30, 56)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

legend_elements = [
    Patch(facecolor=box_colors[0], edgecolor='#2C3E50', alpha=0.75, label='Без шума (n = 9 512)'),
    Patch(facecolor=box_colors[1], edgecolor='#2C3E50', alpha=0.75, label='С шумом (n = 1 833)'),
    plt.Line2D([0], [0], color='#2C3E50', lw=2.2, label='Медиана'),
    plt.Line2D([0], [0], color='#F39C12', lw=2.0, ls='--', label='Выборочное среднее')
]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.99),
           ncol=4, frameon=True, facecolor='#FFFFFF', edgecolor='#BDC3C7')

plt.tight_layout(rect=[0, 0, 1, 0.94])
fig2_path = os.path.join(OUTPUT_DIR, 'fig2_boxplots_distribution_shift_pvalue.png')
plt.savefig(fig2_path)
plt.close()
print(f"Сохранен: {fig2_path}")


Генерация Графика 2: Boxplot распределений пробега и температуры...


Сохранен: pictures/fig2_boxplots_distribution_shift_pvalue.png


### Рис. 3. Эффект большой выборки для фактора пробега

In [7]:
# -----------------------------------------------------------------------------
# ГРАФИК 4: МОДЕЛИРОВАНИЕ "ЛОВУШКИ СВЕРХБОЛЬШОЙ ВЫБОРКИ" (P-VALUE VS N)
# -----------------------------------------------------------------------------
print("Генерация Графика 4: Моделирование ловушки сверхбольшой выборки...")

test_N_range = np.logspace(2.5, 6.0, 300)
prop0 = (N_TOTAL - N_NOISE) / N_TOTAL
prop1 = N_NOISE / N_TOTAL

d_mil_const = 0.04648
d_tmp_const = 0.00264

p_values_mil = []
p_values_tmp = []

for cur_N in test_N_range:
    tn0 = cur_N * prop0
    tn1 = cur_N * prop1
    se_cur = np.sqrt(1.0 / tn0 + 1.0 / tn1)
    
    z_m = d_mil_const / se_cur
    p_m = 2.0 * (1.0 - stats.norm.cdf(abs(z_m)))
    p_values_mil.append(max(p_m, 1e-16))
    
    z_t = d_tmp_const / se_cur
    p_t = 2.0 * (1.0 - stats.norm.cdf(abs(z_t)))
    p_values_tmp.append(max(p_t, 1e-16))

fig, ax = plt.subplots(figsize=(9.8, 6.0))

ax.plot(test_N_range, p_values_mil, label=f'Пробег: сдвиг всего +465 км (Cohen $d = {d_mil_const:.4f}$)',
        color='#C0392B', lw=2.5, zorder=4)
ax.plot(test_N_range, p_values_tmp, label=f'Температура: сдвиг +0.026 °C (Cohen $d = {d_tmp_const:.4f}$)',
        color='#2980B9', lw=2.2, linestyle='-.', zorder=4)

ax.axhline(0.05, color='#2C3E50', linestyle='--', lw=1.5, zorder=3,
           label='Стандартный критический порог $\\alpha = 0.05$')

ax.axhspan(1e-16, 0.05, color='#FDEDEC', alpha=0.6, zorder=1)
ax.text(600, 0.00000000001,
        'ЗОНА ИСКУССТВЕННОЙ СТАТИСТИЧЕСКОЙ ДОСТОВЕРНОСТИ\n(Large Sample Size Fallacy / The p-Value Trap):\nЛюбой ничтожный сдвиг становится «высокозначимым» при $N > 15\,000$',
        fontsize=9.2, color='#900C3F', fontweight='bold',
        bbox=dict(boxstyle='square,pad=0.4', facecolor='#FADBD8', edgecolor='#E6B0AA', alpha=0.92))

ax.axvline(N_TOTAL, color='#27AE60', linestyle=':', lw=1.8, zorder=3)
ax.scatter([N_TOTAL], [0.0644], color='#C0392B', s=70, zorder=5)

# Чистая плашка аннотации без пересечения кривой
ax.annotate(
    f'Текущий объем выборки телематики\n$N = {N_TOTAL:,}$ авт. ($p = 0.064$)\nПограничная зона перехода'.replace(',', ' '),
    xy=(N_TOTAL, 0.0644),
    xytext=(N_TOTAL * 1.5, 0.15),
    arrowprops=dict(facecolor='#2C3E50', shrink=0.08, width=1.2, headwidth=6),
    fontsize=8.8, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.35', facecolor='#FFFFFF', edgecolor='#27AE60', alpha=0.95),
    zorder=6
)

ax.scatter([20000], [0.015], color='#C0392B', s=60, zorder=5)
ax.annotate(
    'При $N > 20\,000$\nформально $p < 0.01$!',
    xy=(20000, 0.015),
    xytext=(45000, 0.001),
    arrowprops=dict(facecolor='#C0392B', shrink=0.08, width=1.2, headwidth=6),
    fontsize=8.8, fontweight='bold', color='#C0392B',
    bbox=dict(boxstyle='round,pad=0.25', facecolor='#FFFFFF', edgecolor='#C0392B', alpha=0.9),
    zorder=6
)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(300, 1000000)
ax.set_ylim(1e-16, 1.5)

ax.set_xlabel('Объем выборки наблюдений ($N$), логарифмическая шкала')
ax.set_ylabel('Расчетный уровень значимости ($p$-value), log-шкала')

ax.legend(loc='lower left', frameon=True, facecolor='#FFFFFF', edgecolor='#BDC3C7')
ax.grid(True, which="both", ls="--")

plt.tight_layout()
fig4_path = os.path.join(OUTPUT_DIR, 'fig4_large_sample_size_fallacy.png')
plt.savefig(fig4_path)
plt.close()
print(f"Сохранен: {fig4_path}")


Генерация Графика 4: Моделирование ловушки сверхбольшой выборки...


<>:41: SyntaxWarning: invalid escape sequence '\,'
<>:61: SyntaxWarning: invalid escape sequence '\,'
<>:41: SyntaxWarning: invalid escape sequence '\,'
<>:61: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipykernel_907/2246835428.py:41: SyntaxWarning: invalid escape sequence '\,'
  'ЗОНА ИСКУССТВЕННОЙ СТАТИСТИЧЕСКОЙ ДОСТОВЕРНОСТИ\n(Large Sample Size Fallacy / The p-Value Trap):\nЛюбой ничтожный сдвиг становится «высокозначимым» при $N > 15\,000$',
/tmp/ipykernel_907/2246835428.py:61: SyntaxWarning: invalid escape sequence '\,'
  'При $N > 20\,000$\nформально $p < 0.01$!',


Сохранен: pictures/fig4_large_sample_size_fallacy.png


### Рис. 4. Технико-экономическое сопоставление управленческих сценариев

In [8]:
# -----------------------------------------------------------------------------
# ГРАФИК 7: ЭКОНОМИЧЕСКОЕ СОПОСТАВЛЕНИЕ И ОХВАТ ПАРКА (ВИЗУАЛЬНЫЙ)
# -----------------------------------------------------------------------------
print("Генерация Графика 7: Экономическая диаграмма и охват автопарка...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.0, 5.8))

# 1. Финансовые затраты
cost_false = 130.1
cost_targeted = 16.8
saved_cost = cost_false - cost_targeted
pct_saved = (saved_cost / cost_false) * 100

categories = ['Сценарий 1\n(поголовная замена)', 'Сценарий 2\n(адресный подход)']
costs = [cost_false, cost_targeted]
colors_cost = ['#D9534F', '#27AE60']

bars1 = ax1.bar(categories, costs, color=colors_cost, width=0.48, edgecolor='#2C3E50', lw=1.2, zorder=3)
for bar, val in zip(bars1, costs):
    ax1.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 3.0,
        f"{val:.1f} млн руб.", ha='center', va='bottom',
        fontweight='bold', fontsize=11.5, color='#2C3E50'
    )

# Визуальная выноска предотвращенного ущерба
ax1.annotate(
    f"Экономия: {saved_cost:.1f} млн руб.\n(снижение на {pct_saved:.1f} %)",
    xy=(1, cost_targeted + 5),
    xytext=(0.42, 85),
    arrowprops=dict(facecolor='#27AE60', shrink=0.08, width=2.0, headwidth=8),
    fontsize=10.5, fontweight='bold', color='#1E8449',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#E8F8F5', edgecolor='#A3E4D7', lw=1.2)
)

ax1.set_ylabel('Финансовые затраты, млн руб.', fontsize=12.0)
ax1.set_ylim(0, 155)
ax1.set_title('(а) Сопоставление финансовых затрат', pad=12, fontweight='bold', fontsize=12.5)
ax1.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)

# 2. Охват парка сервисными воздействиями
fleet_false = 10838
fleet_targeted = 867
ratio_fleet = fleet_false / fleet_targeted

fleet_vals = [fleet_false, fleet_targeted]
colors_fleet = ['#E67E22', '#2980B9']

bars2 = ax2.bar(categories, fleet_vals, color=colors_fleet, width=0.48, edgecolor='#2C3E50', lw=1.2, zorder=3)
for bar, val in zip(bars2, fleet_vals):
    ax2.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 250,
        f"{val:,} авт.".replace(',', ' '), ha='center', va='bottom',
        fontweight='bold', fontsize=11.5, color='#2C3E50'
    )

# Визуальная выноска снижения охвата
ax2.annotate(
    f"Снижение охвата\nв {ratio_fleet:.1f} раза",
    xy=(1, fleet_targeted + 400),
    xytext=(0.42, 6500),
    arrowprops=dict(facecolor='#2980B9', shrink=0.08, width=2.0, headwidth=8),
    fontsize=10.5, fontweight='bold', color='#1B4F72',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#EBF5FB', edgecolor='#AED6F1', lw=1.2)
)

ax2.set_ylabel('Охват автопарка, автомобилей', fontsize=12.0)
ax2.set_ylim(0, 13000)
ax2.set_title('(б) Охват автопарка сервисными работами', pad=12, fontweight='bold', fontsize=12.5)
ax2.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)

plt.tight_layout()
fig7_path = os.path.join(OUTPUT_DIR, 'fig7_decision_matrix_and_economic_impact.png')
plt.savefig(fig7_path, dpi=300)
plt.close()
print(f"Сохранен: {fig7_path}")


Генерация Графика 7: Экономическая диаграмма и охват автопарка...


Сохранен: pictures/fig7_decision_matrix_and_economic_impact.png
